In [1]:
import os
import json
import numpy as np
import torch
import clip
from PIL import Image
from sentence_transformers import SentenceTransformer

/Users/administrator/miniconda3/envs/SpottySearch/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IMG_DIR = "Data"
LABELS_FILE = "labels.json"
CAPTIONS_FILE = "captions.json"
KS = (1, 3, 5)

In [3]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [4]:
def normalise(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

In [5]:
image_files = sorted(
    f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))
)
index_of = {f: i for i, f in enumerate(image_files)}
 
with open(LABELS_FILE) as f:
    labels = json.load(f)
with open(CAPTIONS_FILE) as f:
    captions = json.load(f)
 
# queries: labelled photos that still exist and have a non-empty description
queries = [(f, q) for f, q in labels.items() if f in index_of and q.strip()]
print(f"Evaluating on {len(queries)} labelled queries over a pool of {len(image_files)} photos.\n")

Evaluating on 20 labelled queries over a pool of 43 photos.



In [6]:
clip_model, preprocess = clip.load("ViT-B/32", device=device)
batch = torch.stack(
    [preprocess(Image.open(os.path.join(IMG_DIR, f))) for f in image_files]
).to(device)
with torch.no_grad():
    clip_img = normalise(clip_model.encode_image(batch).cpu().numpy())
 
 
def clip_rank(query, true_idx):
    with torch.no_grad():
        tok = clip.tokenize([query], truncate=True).to(device)
        q = normalise(clip_model.encode_text(tok).cpu().numpy())
    sims = (clip_img @ q.T).squeeze(-1)
    order = np.argsort(sims)[::-1]
    return int(np.where(order == true_idx)[0][0]) + 1   # 1-based rank

In [7]:
text_model = SentenceTransformer("all-MiniLM-L6-v2")
cap_vecs = text_model.encode([captions[f] for f in image_files], normalize_embeddings=True)
 
 
def caption_rank(query, true_idx):
    q = text_model.encode(query, normalize_embeddings=True)
    sims = cap_vecs @ q
    order = np.argsort(sims)[::-1]
    return int(np.where(order == true_idx)[0][0]) + 1

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8147.35it/s]


In [8]:
def evaluate(rank_fn):
    ranks = np.array([rank_fn(q, index_of[f]) for f, q in queries])
    metrics = {f"recall@{k}": float(np.mean(ranks <= k)) for k in KS}
    metrics["MRR"] = float(np.mean(1.0 / ranks))
    metrics["median_rank"] = float(np.median(ranks))
    return metrics, ranks
 
 
clip_metrics, clip_ranks = evaluate(clip_rank)
cap_metrics, cap_ranks = evaluate(caption_rank)

In [9]:
cols = [f"recall@{k}" for k in KS] + ["MRR", "median_rank"]
print(f"{'method':<10}" + "".join(f"{c:>13}" for c in cols))
for name, m in [("CLIP", clip_metrics), ("Caption", cap_metrics)]:
    print(f"{name:<10}" + "".join(f"{m[c]:>13.2f}" for c in cols))
 
print("\nPer-query rank (lower is better, 1 = top result):")
print(f"{'image':<42}{'CLIP':>6}{'Caption':>9}")
for (f, _), cr, pr in sorted(zip(queries, clip_ranks, cap_ranks), key=lambda t: t[1]):
    print(f"{f:<42}{cr:>6}{pr:>9}")

method         recall@1     recall@3     recall@5          MRR  median_rank
CLIP               0.45         0.65         0.70         0.60         2.00
Caption            0.60         0.80         0.80         0.71         1.00

Per-query rank (lower is better, 1 = top result):
image                                       CLIP  Caption
PHOTO-2026-06-02-06-58-12 12.jpg               1        1
PHOTO-2026-06-02-06-58-12 13.jpg               1        1
PHOTO-2026-06-02-06-58-12 16.jpg               1        1
PHOTO-2026-06-02-06-58-12 17.jpg               1        1
PHOTO-2026-06-02-06-58-12 18.jpg               1        1
PHOTO-2026-06-02-06-58-12 2.jpg                1        1
PHOTO-2026-06-02-06-58-12 5.jpg                1        1
PHOTO-2026-06-02-06-58-12 9.jpg                1        1
PHOTO-2026-06-02-06-58-13 10.jpg               1        2
PHOTO-2026-06-02-06-58-12 10.jpg               2        6
PHOTO-2026-06-02-06-58-12 14.jpg               2       26
PHOTO-2026-06-02-06-58-12